# AdaBoost (Adaptive Boosting)
A boosting-based ensemble algorithm that combines multiple weak learners to create a strong predictive model.  
It builds models sequentially, where each new model focuses more on the data points that were misclassified by previous models.

It starts with a simple model (usually a shallow decision tree, often called a decision stump).  
Instead of treating all data points equally, it assigns weights to each training sample.

Initially, all the samples have equal weight, but after each iteration, misclassified samples are given higher weights and correctly classified samples are given lower weights.  

This encourages the next model to focus more on difficult examples.

## Model formulation

The final prediction is a weighted combination of weak learners:

$$
F(x) = \sum_{t=1}^{T} \alpha_t h_t(x)
$$

- $h_t(x)$ → weak learner (e.g., decision stump)  
- $\alpha_t$ → weight assigned to each learner  
- $T$ → number of iterations  

Each model contributes based on how well it performs.

## Training process

The algorithm proceeds step by step:

1. initialize sample weights equally  
2. train a weak learner on the weighted data  
3. compute the error of the learner  
4. assign a weight to the learner based on its error  
5. update sample weights (increase for misclassified points)  
6. repeat for multiple iterations  

## Error and model weight

For a given weak learner, the weighted error is:

$$
\epsilon_t = \sum_{i=1}^{n} w_i \cdot \mathbb{1}(y_i \neq h_t(x_i))
$$

The model weight is then computed as:

$$
\alpha_t = \frac{1}{2} \log \left(\frac{1 - \epsilon_t}{\epsilon_t}\right)
$$

- if error is low → $\alpha_t$ is high (the model is trusted more)  
- if error is high → $\alpha_t$ is low  

## Updating sample weights

After each iteration, sample weights are updated:

$$
w_i \leftarrow w_i \cdot \exp(-\alpha_t y_i h_t(x_i))
$$

The weights are then normalized so that they sum to 1.

## Final prediction (classification)

For classification, the final prediction is:

$$
\hat{y} = \text{sign}\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right)
$$

This combines all weak learners into a strong classifier.

-----

# AdaBoost for Regression

In regression tasks, the model focuses on the magnitude of prediction errors rather than misclassification.

### Prediction 

The final prediction is a weighted combination of weak learners:

$$
\hat{y} = \sum_{t=1}^{T} \alpha_t h_t(x)
$$

Here, each $h_t(x)$ predicts a continuous value.

### Error computation

For regression, the error is based on absolute or squared differences:

$$
e_i = |y_i - h_t(x_i)|
$$

This error is normalized to lie between 0 and 1 before being used for weight updates.

### Model weight 

The model weight is computed as:

$$
\alpha_t = \log \left(\frac{1}{\beta_t}\right)
$$

where:

$$
\beta_t = \frac{\epsilon_t}{1 - \epsilon_t}
$$

and $\epsilon_t$ is the normalized error.

### Weight update 

Sample weights are updated based on the magnitude of the error:

$$
w_i \leftarrow w_i \cdot \beta_t^{(1 - e_i)}
$$

Larger errors result in higher weights, which ensures that the next model focuses more on poorly predicted samples.

### Loss function 

AdaBoost regression can use different loss functions such as:

- linear loss  
- square loss  
- exponential loss  

These determine how strongly larger errors are penalized.

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import AdaBoostRegressor
from sklearn.preprocessing import StandardScaler
cali=fetch_california_housing()
X=cali.data
y=cali.target

X_tr, X_test, y_tr, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ada=AdaBoostRegressor(n_estimators=200,learning_rate=0.01, random_state=42)
ada.fit(X_tr,y_tr)
y_pred=ada.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R^2 Score: {r2_score(y_test, y_pred):.4f}")


MAE: 0.58
MSE: 0.58
R^2 Score: 0.5582


In [21]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_wine
from sklearn.ensemble import AdaBoostClassifier
wine=load_wine()
X=wine.data
y=wine.target
Xtr, X_test, ytr, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ada_clf=AdaBoostClassifier(n_estimators=100, learning_rate=0.09
, random_state=42)
ada_clf.fit(Xtr,ytr)
y_pred=ada_clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))


Accuracy: 0.89
Classification Report:
              precision    recall  f1-score   support

     class_0       0.93      1.00      0.97        14
     class_1       0.92      0.79      0.85        14
     class_2       0.78      0.88      0.82         8

    accuracy                           0.89        36
   macro avg       0.88      0.89      0.88        36
weighted avg       0.89      0.89      0.89        36



# AdaBoost Parameters(Crucial ones)

1. **n_estimators**  
This defines the number of weak learners (typically decision trees) that are trained sequentially.  (too small means underfit, too large causes overfit risk)



2. **learning_rate** 
    controls how much each weak learner contributes to the final model  

   
   In theory, the final prediction is:

    $F(x)=∑α_th_t(x)$  
    
    where $\alpha_t$ is computed automatically based on the learner’s error.
     
     But in practice, the learning rate scales the contribution of each learner:

    $$
    F(x) = \sum \eta \cdot \alpha_t h_t(x)
    $$

    where $\alpha_t$ is automatically computed based on the learner’s error, and $\eta$ (learning_rate) scales this contribution.


    Even though $\alpha_t$ determines the importance of each learner, the learning rate controls how strongly all learners affect the final model.


3. **estimator (previously base_estimator)**  
   This specifies the base learner used in boosting (default is a decision tree).  
   The choice of estimator determines how complex each individual model is.


4. **loss (Regressor)**  
   This defines the loss function used to measure error in regression tasks.

   Options:  
   - `'linear'`  
   - `'square'`  
   - `'exponential'`  

   It controls how strongly errors influence the update of sample weights, with higher penalties emphasizing larger errors more.



5. **random_state**  
   This controls randomness in the model (such as sampling and tree construction).  

   Effect:  
   - ensures reproducibility of results  
   - does not directly affect model performance  


6. **algorithm (Classifier)**  
    Used in `AdaBoostClassifier`.

    Options:  
    - `'SAMME'` 
    - `'SAMME.R'`

    In SAMME(discrete boosting), each weak learner contributes based only on whether predictions are correct or incorrect, and the model assigns weights using classification error.

    In SAMME.R (real-valued boosting)  , the model uses predicted class probabilities instead of hard labels, allowing it to take into account how confident each prediction is, leading to smoother updates and typically faster convergence.

    In practice, `'SAMME.R'` is preferred because it makes better use of the information provided by the base learner.

7. **estimator__max_depth (indirect parameter)**  
   When using decision trees as base learners, this parameter controls how deep each tree can grow.  

   - smaller depth → weaker learners (preferred for boosting)
   - larger depth → stronger learners but higher risk of overfitting  
# Docs :
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html  
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html